# 2일차 파트 2 실습 풀이

`2일차_02_실습.ipynb`의 풀이와 해설입니다. 풀이는 하나의 예시이며, 테스트 셀을 통과한다면 다르게 작성해도 됩니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 받지 않을 때만 바꿉니다(파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. Windows PowerShell에서는 다음과 같이 설정한 뒤 터미널을 새로 엽니다.

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. 이 값이 없으면 노트북은 Anthropic API를 쓰는 것으로 봅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 마지막 줄에 `준비 완료`가 나오면 연결된 것입니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/2일차'
SYSTEM = open(f'{DATA}/분류_규칙.txt', encoding='utf-8').read()

# 100만 토큰당 달러 (입력, 출력). Anthropic 공개 가격, 2026-11-03 확인.
# 사내 API 를 쓴다면 사내 과금 기준으로 바꿉니다. 과금이 없으면 (0, 0) 으로 둡니다.
PRICE = {'소형': (0.10, 0.50), '기본': (2.00, 10.00), '대형': (4.00, 20.00)}


def cost_usd(input_tokens, output_tokens, model='기본'):
    """토큰 수와 단가로 요금(달러)을 계산합니다."""
    input_price, output_price = PRICE[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ... 가 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `401 auth` | 키가 틀렸습니다. 앞뒤 공백이나 따옴표 없이 다시 넣습니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `NameError` | 앞 셀을 건너뛰었습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 준비

아래 셀은 세 실습이 함께 쓰는 재료와 함수입니다. 그대로 실행합니다.

| 이름 | 내용 |
| --- | --- |
| `tickets` | 접수 80건. 열은 `id`, `접수시각`, `본문`입니다 |
| `ticket_answers` | 정답 라벨. 열은 `team`, `location`, `urgent`, `난이도`입니다. 위치가 없는 접수의 `location`은 `None`입니다 |
| `TICKET_RULES` | 담당팀 기준(`접수_규칙.txt`). 시스템 프롬프트로 씁니다 |
| `prompt_of(row)` | 접수 한 건을 모델에 보낼 글로 만듭니다 |
| `parse_reply(reply, model_class)` | 응답을 검증해 `(객체, None)` 또는 `(None, 사유)`를 돌려줍니다. 교재의 예제 2-5와 같습니다 |
| `open_db(path)` | `requests` 테이블이 있는 SQLite DB를 엽니다. `id`가 기본 키입니다 |

In [3]:
# 실습 준비
import sqlite3
from typing import Literal

from pydantic import BaseModel, ConfigDict, ValidationError

import checks2

tickets = pd.read_csv(f'{DATA}/접수_실습.csv').fillna('')
ticket_answers = pd.read_csv(f'{DATA}/접수_실습_정답.csv', dtype=str, keep_default_na=False).set_index('id')
ticket_answers['urgent'] = ticket_answers['urgent'] == 'true'
ticket_answers['location'] = ticket_answers['location'].map(lambda value: value or None)
TICKET_RULES = open(f'{DATA}/접수_규칙.txt', encoding='utf-8').read().strip()
print(TICKET_RULES)
print('----')
print(tickets.head(8).to_string(index=False))


def prompt_of(row):
    """접수 한 건을 모델에 보낼 글로 만듭니다."""
    return f"id: {row['id']}\n본문: {row['본문']}"


def parse_reply(reply, model_class):
    """응답을 검증해 (객체, None) 또는 (None, 사유) 를 돌려줍니다."""
    if reply.stop_reason != 'end':
        return None, {'max_tokens': '잘림', 'refusal': '거절'}.get(reply.stop_reason, '비정상 종료')
    text = reply.text[reply.text.find('{'):reply.text.rfind('}') + 1]
    try:
        return model_class.model_validate_json(text), None
    except ValidationError as error:
        first = error.errors()[0]
        field = '.'.join(str(part) for part in first['loc']) or '전체'
        return None, f"스키마 불일치({field}: {first['type']})"


def open_db(path=':memory:'):
    """requests 테이블이 있는 DB 를 엽니다. 경로를 주지 않으면 메모리에만 있는 DB 입니다."""
    db = sqlite3.connect(path)
    db.execute('''CREATE TABLE IF NOT EXISTS requests (
        id TEXT PRIMARY KEY, team TEXT, location TEXT, urgent INTEGER,
        model TEXT, run_at TEXT, reviewed TEXT)''')
    return db

너는 반도체 공장에 들어오는 접수 요청을 담당팀으로 나누는 도우미다.

담당팀(team)은 아래 다섯 가지 중 하나다.
- 설비: 생산 장비의 고장, 알람, 점검, 부품 교체
- 시설: 건물, 공조, 전기, 수도, 조명, 사무 집기와 편의 시설. 초순수와 전력 같은 유틸리티 공급도 시설이다
- IT: PC, 계정, 네트워크, 서버, 사내 시스템, 프린터
- 안전: 사람의 안전, 화학물질과 가스 누출, 보호구, 화재, 출입 보안
- 기타: 위 네 팀의 일이 아닌 요청이나 요청이 아닌 글

두 팀에 걸치면 안전, 설비, 시설, IT 순으로 앞선 것을 고른다.
location 은 본문에 나온 장비 이름이나 장소를 본문의 글자 그대로 옮긴 것이다. 본문에 없으면 null 이다.
urgent 는 생산이 멈췄거나 사람이 다칠 수 있는 상황이면 true, 아니면 false 다.
----
  id             접수시각                                                본문
R001 2026-11-09 01:14                        통근 버스 노선에 정류장을 하나 추가해 주세요.
R002 2026-11-09 02:28                       다음 달 연차 사용 계획을 어디에 올리면 되나요.
R003 2026-11-09 04:20                              오늘 회식 장소 아시는 분 계신가요.
R004 2026-11-09 05:00        SPT-03 타깃 수명이 90%를 넘었습니다. 교체 계획 확인 부탁드립니다.
R005 2026-11-09 06:22      엘리베이터 2호기가 3층에서 멈춘 채 움직이지 않습니다. 안에 사람은 없습니다.
R006 2026-11-09 06:37                     출입 카드를 잃어버렸습니다. 정지 처리 부탁드립니다.
R007 2026-11-09 10:40                               볼펜과 포

## 실습 5 스키마 설계

**목적** 결과 테이블의 열에서 출발해 스키마를 정합니다. 허용값과, 값이 없을 때의 처리를 직접 정해 봅니다.

**할 일** pydantic 클래스 `Ticket`을 작성합니다. 결과 테이블에 필요한 열은 네 개입니다.

| 필드 | 내용 |
| --- | --- |
| `id` | 접수 id |
| `team` | 담당팀. 설비, 시설, IT, 안전, 기타 가운데 하나 |
| `location` | 본문에 나온 장비 이름이나 장소. 본문에 없으면 값이 없습니다 |
| `urgent` | 긴급 여부 |

**완료 기준** 테스트 셀이 "실습 5 통과"를 출력합니다. 그다음 3건을 실제로 보내, 위치가 없는 접수의 `location`이 `None`으로 오는지 확인합니다.

<details><summary>힌트 1</summary>

허용값은 `Literal['설비', '시설', ...]`로, 없을 수 있는 값은 `str | None`으로 적습니다.
</details>

<details><summary>힌트 2</summary>

`location: str | None = None`처럼 기본값을 주면 그 필드는 필수가 아니게 됩니다. 구조화 출력의 스키마는 모든 필드가 필수여야 하므로 기본값을 주지 않습니다. 값이 없다는 것은 필드를 빼는 것이 아니라 `null`을 넣는 것으로 표현합니다.
</details>

<details><summary>힌트 3</summary>

정하지 않은 필드를 거부하고 문자열을 불리언 값으로 바꿔 받지 않게 하려면 `model_config = ConfigDict(extra='forbid', strict=True)`를 클래스 안에 적습니다.
</details>

In [4]:
# 실습 5
class Ticket(BaseModel):
    """접수 한 건의 결과. 필드는 id, team, location, urgent 입니다."""
    model_config = ConfigDict(extra='forbid', strict=True)
    id: str
    team: Literal['설비', '시설', 'IT', '안전', '기타']
    location: str | None          # 기본값을 주지 않습니다. 없으면 null 이 와야 합니다
    urgent: bool

In [5]:
# 실습 5 확인 (API 를 부르지 않습니다)
checks2.check_schema(Ticket)

1. 통과 올바른 응답을 받아들인다
2. 통과 location 이 null 인 응답을 받아들인다
3. 통과 목록에 없는 team 을 거부한다
4. 통과 location 이 빠진 응답을 거부한다
5. 통과 불리언 값 자리의 문자열을 거부한다
6. 통과 정하지 않은 필드를 거부한다
7. 통과 스키마가 구조화 출력의 두 조건을 지킨다
실습 5 통과


True

In [6]:
# 실습 5 실행 (실제 API 를 부릅니다. 3건)
for _, row in tickets.iloc[[3, 5, 7]].iterrows():
    reply = llm.ask(prompt_of(row), system=TICKET_RULES, model='소형', schema=Ticket.model_json_schema())
    parsed, reason = parse_reply(reply, Ticket)
    print(row['본문'][:40])
    print('  →', parsed, '· 사유:', reason)

SPT-03 타깃 수명이 90%를 넘었습니다. 교체 계획 확인 부탁드립니
  → id='R004' team='설비' location='SPT-03' urgent=False · 사유: None


출입 카드를 잃어버렸습니다. 정지 처리 부탁드립니다.
  → id='R006' team='안전' location=None urgent=False · 사유: None


ETCH-02 옆 바닥에 약액이 흘러 있습니다. 미끄러져 다칠 위험이 있
  → id='R008' team='안전' location='ETCH-02' urgent=True · 사유: None


**해설**

- `location`을 `str`로만 두면 위치가 없는 접수에서도 모델이 문자열을 채워야 합니다. `str | None`으로 두면 "없음"을 `None` 한 가지로 받을 수 있고, 실습 7에서 "본문에 없는 위치를 적었는가"를 검사하기도 쉬워집니다.
- 기본값(`= None`)을 주면 pydantic은 그 필드를 스키마의 `required`에서 뺍니다. 그러면 모델이 필드를 아예 빼고 응답해도 검증을 통과합니다. "값이 없다"와 "응답에서 빠졌다"를 구분하려면 필드는 필수로 두고 `null`을 허용합니다.
- `team`의 목록에 "기타"가 있어야 네 팀의 일이 아닌 접수를 받을 곳이 생깁니다. 실습 7의 실험에서 목록을 줄이면 어떤 일이 생기는지 확인합니다.

## 실습 6 배치 처리와 멱등한 저장

**목적** 한 건의 실패가 배치를 멈추지 않게 하고, 다시 실행해도 행이 중복되지 않으며 이미 처리한 건을 다시 보내지 않게 합니다.

**할 일** 함수 세 개를 작성합니다.

1. `classify(row, ask=llm.ask, model_class=None, rules=None)`: 접수 한 건을 보내고 dict를 돌려줍니다. 어떤 경우에도 예외를 내지 않습니다.
    - `ask(prompt_of(row), system=규칙, model='소형', max_tokens=1024, schema=스키마)`로 보냅니다. `model_class`가 `None`이면 `Ticket`을, `rules`가 `None`이면 `TICKET_RULES`를 씁니다(실습 7의 실험에서 다른 값을 넘깁니다).
    - 돌려주는 dict의 키는 `id`, `answer_id`, `team`, `location`, `urgent`, `error`, `input_tokens`, `output_tokens`입니다. `id`는 보낸 id, `answer_id`는 모델이 돌려준 id입니다.
    - 성공하면 `error`는 `None`입니다. 호출 실패(`llm.CallError`)나 `parse_reply()`가 돌려준 사유가 있으면 `error`에 적고 `team`, `location`, `urgent`, `answer_id`는 `None`으로 둡니다.
2. `save(db, records, model, run_at)`: `error`가 없는 결과만 `requests` 테이블에 넣고 넣은 건수를 돌려줍니다. 같은 `id`가 있으면 `team`, `location`, `urgent`, `model`, `run_at`만 고칩니다. `reviewed` 열은 건드리지 않습니다.
3. `run(db, rows, ask=llm.ask)`: `requests` 테이블에 없는 `id`의 행만 `classify()`로 보내고 `save()`로 저장합니다. `(보낸 건수, 실패 목록)`을 돌려줍니다. 실패 목록은 `[(id, 사유), ...]`입니다.

**완료 기준** 테스트 셀이 "실습 6 통과"를 출력합니다. 그다음 80건을 실제로 처리해 저장한 행 수를 확인하고, 한 번 더 실행했을 때 보내는 건이 0건인지 확인합니다.

<details><summary>힌트 1</summary>

`classify()`는 결과 dict를 먼저 만들어 두고(`error`와 결과 필드는 `None`), 단계마다 채워 나가면 어느 경로에서든 같은 키를 가진 dict를 돌려줄 수 있습니다. `except`에는 `llm.CallError`만 적습니다.
</details>

<details><summary>힌트 2</summary>

`save()`의 SQL은 `INSERT INTO requests (id, team, location, urgent, model, run_at) VALUES (?, ?, ?, ?, ?, ?) ON CONFLICT(id) DO UPDATE SET team = excluded.team, ...`처럼 작성합니다. `INSERT OR REPLACE`를 쓰면 `reviewed` 열이 지워져 마지막 테스트를 통과하지 못합니다.
</details>

<details><summary>힌트 3</summary>

`run()`은 먼저 `SELECT id FROM requests`로 저장된 id의 집합을 만들고, 거기에 없는 행만 골라 보냅니다. 여러 건은 `ThreadPoolExecutor`로 병렬 처리하되, DB에 쓰는 일은 병렬 처리가 끝난 뒤 한 번에 합니다.
</details>

In [7]:
# 실습 6
def classify(row, ask=llm.ask, model_class=None, rules=None):
    """접수 한 건을 보내 결과 dict 를 돌려줍니다. 실패해도 예외를 내지 않습니다."""
    model_class = model_class or Ticket
    record = {'id': row['id'], 'answer_id': None, 'team': None, 'location': None, 'urgent': None,
              'error': None, 'input_tokens': 0, 'output_tokens': 0}
    try:
        reply = ask(prompt_of(row), system=rules or TICKET_RULES, model='소형', max_tokens=1024,
                    schema=model_class.model_json_schema())
    except llm.CallError as error:            # API 호출의 실패만 잡습니다
        record['error'] = f'호출 실패({error.kind})'
        return record
    record.update(input_tokens=reply.input_tokens or 0, output_tokens=reply.output_tokens or 0)
    parsed, reason = parse_reply(reply, model_class)
    if reason:
        record['error'] = reason
    else:
        record.update(answer_id=parsed.id, team=parsed.team, location=parsed.location, urgent=parsed.urgent)
    return record


UPSERT = '''INSERT INTO requests (id, team, location, urgent, model, run_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                team = excluded.team, location = excluded.location, urgent = excluded.urgent,
                model = excluded.model, run_at = excluded.run_at'''


def save(db, records, model, run_at):
    """error 가 없는 결과를 requests 테이블에 넣고 넣은 건수를 돌려줍니다. 같은 id 는 결과 열만 고칩니다."""
    db_rows = [(record['id'], record['team'], record['location'], int(record['urgent']), model, run_at)
               for record in records if not record['error']]
    db.executemany(UPSERT, db_rows)
    db.commit()
    return len(db_rows)


def run(db, rows, ask=llm.ask):
    """requests 테이블에 없는 id 만 보내고 저장합니다. (보낸 건수, 실패 목록) 을 돌려줍니다."""
    done_ids = {row_id for (row_id,) in db.execute('SELECT id FROM requests')}
    to_send = [row for row in rows if row['id'] not in done_ids]
    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(pool.map(lambda row: classify(row, ask=ask), to_send))
    save(db, records, llm.model_name('소형'), pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
    return len(to_send), [(record['id'], record['error']) for record in records if record['error']]

In [8]:
# 실습 6 확인 (mock 호출을 씁니다)
checks2.check_batch(classify, save, run, open_db)

1. 통과 성공한 건은 결과를 채우고 error 는 None 이다
2. 통과 호출 실패는 예외를 내지 않고 error 에 사유를 적는다
3. 통과 잘린 응답은 error 에 사유를 적는다
4. 통과 스키마에 어긋난 응답은 error 에 사유를 적는다
5. 통과 한 건이 실패해도 끝까지 처리하고 성공한 건만 저장한다
6. 통과 다시 실행하면 저장하지 않은 건만 보낸다
7. 통과 모두 저장한 뒤에는 보내는 건이 없다
8. 통과 다시 저장해도 행이 늘지 않고 사람이 적은 값이 남는다
실습 6 통과


True

In [9]:
# 실습 6 실행 (실제 API 를 부릅니다. 80건)
os.makedirs('out', exist_ok=True)
if os.path.exists('out/requests.db'):
    os.remove('out/requests.db')              # 처음부터 다시 해 볼 수 있게 지웁니다
db = open_db('out/requests.db')
rows = [row for _, row in tickets.iterrows()]

sent, failed = run(db, rows)
print('첫 실행: 보낸 건', sent, '· 실패', failed)
print('저장한 행', db.execute('SELECT COUNT(*) FROM requests').fetchone()[0])
sent, failed = run(db, rows)
print('다시 실행: 보낸 건', sent, '· 실패', failed)
print('저장한 행', db.execute('SELECT COUNT(*) FROM requests').fetchone()[0])
print(db.execute('SELECT team, COUNT(*) FROM requests GROUP BY team ORDER BY team').fetchall())
db.close()

첫 실행: 보낸 건 80 · 실패 []
저장한 행 80
다시 실행: 보낸 건 0 · 실패 []
저장한 행 80
[('IT', 16), ('기타', 10), ('설비', 21), ('시설', 18), ('안전', 15)]


**해설**

- `classify()`가 잡는 예외는 `llm.CallError`뿐입니다. `except Exception`으로 넓히면 함수 안의 오타나 잘못된 키 이름으로 생긴 오류도 "호출 실패"로 기록되고, 80건이 모두 같은 버그로 실패해도 배치는 정상적으로 끝난 것처럼 보입니다.
- `run()`은 DB에 저장된 id를 "처리가 끝난 건"의 기준으로 씁니다. 실패한 건은 저장하지 않으므로 다음 실행에서 자동으로 다시 보냅니다. 따로 실패 목록 파일을 관리하지 않아도 이어서 실행할 수 있습니다.
- 병렬 처리는 API 호출까지만 하고, DB에는 호출이 모두 끝난 뒤 한 번에 씁니다. SQLite의 연결은 만든 스레드에서만 쓸 수 있고, 한 번에 쓰는 편이 빠릅니다. 다만 이 방식은 배치가 중간에 죽으면 그때까지의 결과를 잃습니다. 건수가 많다면 수십 건 단위로 끊어 저장합니다.
- `ON CONFLICT(id) DO UPDATE SET`은 적어 준 열만 고칩니다. 담당자가 `reviewed` 열에 적은 내용은 배치를 다시 실행해도 남습니다.

## 실습 7 검증, 상태, 정확도

**목적** 스키마 검증을 통과한 결과를 얼마나 믿을 수 있는지 숫자로 확인합니다. 그리고 허용값과 기준 문장이 결과를 어떻게 바꾸는지 실험합니다.

**할 일 1** 함수 두 개를 작성합니다. `record`는 `classify()`가 돌려준 dict이고 `text`는 그 접수의 본문입니다.

1. `validate(record, text)`: 걸린 사유(문자열)를, 통과하면 `None`을 돌려줍니다. 아래 순서로 검사합니다.
    - `error`가 있으면 그 값을 그대로 돌려줍니다.
    - 보낸 `id`와 모델이 돌려준 `answer_id`가 다르면 걸립니다.
    - `team`이 "기타"인데 `urgent`가 `True`이면 걸립니다. 네 팀의 일이 아닌 글이 긴급일 수는 없습니다.
    - `location`이 `None`이 아닌데 본문에 그 글자가 없으면 걸립니다. 규칙은 위치를 "본문의 글자 그대로" 옮기라고 했습니다.
2. `status_of(record, text)`: `error`가 있으면 `'manual'`, 그 밖의 사유로 걸리면 `'review'`, 통과하면 `'auto'`를 돌려줍니다.

**할 일 2** 측정 셀을 실행해 기본 실행의 숫자를 확인합니다.

**할 일 3** 실험 A와 실험 B의 셀을 실행하고, 바뀐 숫자와 그 이유를 적습니다.

**완료 기준** 테스트 셀이 "실습 7 통과"를 출력합니다. 스키마 검증 통과율, 업무 규칙에 걸린 건수, 담당팀 정확도, `auto` 가운데 틀린 건수를 적습니다. 두 실험에서 바뀐 숫자와 이유를 한 문장씩 적습니다.

<details><summary>힌트 1</summary>

본문에 그 글자가 있는지는 `record['location'] in text`로 확인합니다. `None`에는 `in`을 쓸 수 없으므로 `None`인지를 먼저 확인합니다.
</details>

<details><summary>힌트 2</summary>

`status_of()`는 `validate()`를 불러 쓰면 됩니다. `error`가 있는 경우만 먼저 가려 `'manual'`로 돌려줍니다.
</details>

In [10]:
# 실습 7
def validate(record, text):
    """결과 한 건을 검사해 걸린 사유를, 통과하면 None 을 돌려줍니다."""
    if record['error']:
        return record['error']
    if record['answer_id'] != record['id']:
        return 'id 불일치'
    if record['team'] == '기타' and record['urgent']:
        return '기타인데 urgent 가 true'
    if record['location'] is not None and record['location'] not in text:
        return '본문에 없는 위치'
    return None


def status_of(record, text):
    """'auto', 'review', 'manual' 가운데 하나를 돌려줍니다."""
    if record['error']:
        return 'manual'                    # 모델의 결과가 없습니다
    return 'review' if validate(record, text) else 'auto'

In [11]:
# 실습 7 확인 (API 를 부르지 않습니다)
checks2.check_validate(validate, status_of)

1. 통과 문제가 없는 결과는 통과한다
2. 통과 위치가 None 인 결과는 통과한다
3. 통과 error 가 있으면 그 사유를 돌려준다
4. 통과 보낸 id 와 받은 id 가 다르면 걸린다
5. 통과 기타인데 urgent 가 true 면 걸린다
6. 통과 본문에 없는 위치는 걸린다
실습 7 통과


True

In [12]:
# 실습 7 측정 (실제 API 를 부릅니다. 80건)
rows = [row for _, row in tickets.iterrows()]
text_of = dict(zip(tickets['id'], tickets['본문']))


def run_all(model_class=None, rules=None):
    """80건을 모두 분류해 결과 dict 의 목록을 돌려줍니다. DB 에는 저장하지 않습니다."""
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda row: classify(row, model_class=model_class, rules=rules), rows))


def report(records, label):
    """검증 결과와 정답 라벨 비교를 출력합니다."""
    done = [record for record in records if not record['error']]
    team_ok = sum(record['team'] == ticket_answers.loc[record['id'], 'team'] for record in done)
    urgent_ok = sum(record['urgent'] == ticket_answers.loc[record['id'], 'urgent'] for record in done)
    statuses = [status_of(record, text_of[record['id']]) for record in records]
    reasons = pd.Series([validate(record, text_of[record['id']]) for record in records]).dropna()
    auto_wrong = [record['id'] for record, status in zip(records, statuses)
                  if status == 'auto' and record['team'] != ticket_answers.loc[record['id'], 'team']]
    print(f'[{label}] 스키마 검증 통과 {len(done)}/{len(records)}'
          f' · 담당팀 정확도 {team_ok}/{len(done)} · 긴급 여부 정확도 {urgent_ok}/{len(done)}')
    print('  상태:', pd.Series(statuses).value_counts().to_dict(), '· 걸린 사유:', reasons.value_counts().to_dict())
    print(f'  auto {statuses.count("auto")}건 가운데 담당팀이 틀린 건 {len(auto_wrong)}건', auto_wrong)
    print('  urgent 가 true 인 건', sum(bool(record['urgent']) for record in done),
          f"(정답은 {int(ticket_answers['urgent'].sum())}건)")


base = run_all()
report(base, '기본')
for record in base:
    answer = ticket_answers.loc[record['id']]
    why = validate(record, text_of[record['id']])
    if why or record['team'] != answer['team']:
        print(f"  {record['id']} 모델 {record['team']} / {record['location']!r} · 정답 {answer['team']}"
              f" · 사유 {why} · {text_of[record['id']][:30]}")

[기본] 스키마 검증 통과 80/80 · 담당팀 정확도 76/80 · 긴급 여부 정확도 79/80
  상태: {'auto': 80} · 걸린 사유: {}
  auto 80건 가운데 담당팀이 틀린 건 4건 ['R007', 'R036', 'R038', 'R073']
  urgent 가 true 인 건 22 (정답은 21건)
  R007 모델 시설 / None · 정답 기타 · 사유 None · 볼펜과 포스트잇 구매 요청드립니다.
  R036 모델 설비 / 'B라인' · 정답 시설 · 사유 None · 방금 순간 정전이 있었고 B라인 장비가 전부 꺼졌습니다
  R038 모델 시설 / None · 정답 기타 · 사유 None · 구내 식단에 채식 메뉴를 추가해 주실 수 있나요.
  R073 모델 안전 / '세정 장비 3호기' · 정답 설비 · 사유 None · 세정 장비 3호기 배수 밸브가 닫히지 않습니다. 약액이


In [13]:
# 실습 7 실험 A (담당팀 허용값에서 IT 를 뺍니다. 80건을 다시 보냅니다)
class TicketNoIT(Ticket):
    team: Literal['설비', '시설', '안전', '기타']


no_it = run_all(model_class=TicketNoIT)
report(no_it, 'IT 제외')
it_ids = set(ticket_answers.index[ticket_answers['team'] == 'IT'])
moved = pd.Series([record['team'] for record in no_it if record['id'] in it_ids])
print(f'  정답이 IT 인 {len(it_ids)}건이 간 곳:', moved.value_counts(dropna=False).to_dict())

[IT 제외] 스키마 검증 통과 80/80 · 담당팀 정확도 62/80 · 긴급 여부 정확도 79/80
  상태: {'auto': 78, 'review': 2} · 걸린 사유: {'기타인데 urgent 가 true': 2}
  auto 78건 가운데 담당팀이 틀린 건 16건 ['R023', 'R026', 'R030', 'R035', 'R038', 'R041', 'R042', 'R044', 'R047', 'R053', 'R057', 'R058', 'R067', 'R070', 'R073', 'R078']
  urgent 가 true 인 건 22 (정답은 21건)
  정답이 IT 인 16건이 간 곳: {'기타': 15, '시설': 1}


In [14]:
# 실습 7 실험 B (규칙에서 긴급 기준 문장을 뺍니다. 80건을 다시 보냅니다)
URGENT_LINE = next(line for line in TICKET_RULES.splitlines() if line.startswith('urgent'))
print('뺀 문장:', URGENT_LINE)
no_rule = run_all(rules=TICKET_RULES.replace(URGENT_LINE, '').strip())
report(no_rule, '긴급 기준 제외')
flipped = [(new, old) for new, old in zip(no_rule, base) if new['urgent'] != old['urgent']]
print('  긴급 판단이 바뀐 건', len(flipped))
for new, old in flipped[:10]:
    print(f"  {new['id']} 기본 {old['urgent']} → {new['urgent']}"
          f" · 정답 {ticket_answers.loc[new['id'], 'urgent']} · {text_of[new['id']][:34]}")

뺀 문장: urgent 는 생산이 멈췄거나 사람이 다칠 수 있는 상황이면 true, 아니면 false 다.


[긴급 기준 제외] 스키마 검증 통과 80/80 · 담당팀 정확도 78/80 · 긴급 여부 정확도 77/80
  상태: {'auto': 80} · 걸린 사유: {}
  auto 80건 가운데 담당팀이 틀린 건 2건 ['R036', 'R073']
  urgent 가 true 인 건 24 (정답은 21건)
  긴급 판단이 바뀐 건 4
  R006 기본 False → True · 정답 False · 출입 카드를 잃어버렸습니다. 정지 처리 부탁드립니다.
  R033 기본 False → True · 정답 False · 창고 C 셔터가 중간에서 멈췄습니다. 자재를 넣고 빼지 못하고
  R034 기본 True → False · 정답 False · 비상 샤워기 점검표가 두 달째 비어 있습니다. 위치는 세정실 
  R061 기본 False → True · 정답 False · 주차장 차단기가 올라가지 않아 출근 차량이 밀려 있습니다.


**해설**

아래 숫자는 2026-11-19에 Claude 모델로 실행한 결과입니다. 여러분의 숫자는 조금 다를 수 있습니다.

| 항목 | 기본 | 실험 A (IT 제외) | 실험 B (긴급 기준 제외) |
| --- | --- | --- | --- |
| 스키마 검증 통과 | 80/80 | 80/80 | 80/80 |
| 업무 규칙에 걸린 건 | 0 | 2 | 0 |
| 담당팀 정확도 | 77/80 | 62/80 | 76/80 |
| 긴급 여부 정확도 | 80/80 | 80/80 | 77/80 |
| `auto` 가운데 담당팀이 틀린 건 | 80건 중 3건 | 78건 중 16건 | 80건 중 4건 |

기본 실행에서는 80건이 모두 `auto`였고 그 가운데 3건의 담당팀이 틀렸습니다. 틀린 3건은 키오스크 화면(IT인데 시설로), 정전으로 꺼진 장비(시설인데 설비로), 약액이 빠지는 세정 장비(설비인데 안전으로)로, 모두 두 팀에 걸친 접수입니다. 업무 규칙은 이 3건을 하나도 잡지 못했습니다.

- **스키마 검증 통과율은 세 경우 모두 같습니다.** 허용값에서 "IT"를 빼도, 긴급 기준을 빼도 응답의 형식은 스키마에 맞습니다. 통과율만 보면 세 실행을 구분할 수 없습니다. 차이는 정답 라벨과 비교해야 드러납니다.
- **실험 A: 허용값이 답의 범위를 정합니다.** 정답이 IT인 16건 가운데 13건이 기타로, 3건이 시설로 갔습니다. 오류는 한 건도 나지 않았고 담당팀 정확도는 77건에서 62건으로 내려갔습니다. 모델은 목록 안에서 반드시 하나를 골라야 하므로, 맞는 값이 목록에 없으면 가장 가까운 값으로 보냅니다. 업무 규칙("기타인데 urgent가 true")에는 2건만 걸렸고, 틀린 건의 대부분은 `auto`로 지나갔습니다. 새 종류의 접수가 들어올 수 있는 업무라면 "기타"로 분류된 건을 정기적으로 읽어 보고 허용값을 고쳐야 합니다.
- **실험 B: 스키마는 값의 기준을 정하지 않습니다.** `urgent`가 `true`인 건이 21건에서 24건으로 늘었고 긴급 여부 정확도는 80건에서 77건으로 내려갔습니다. 바뀐 3건은 출입 카드 분실, 비상 샤워기 점검표, 화재 감지기 오작동으로 모두 안전과 관련된 접수입니다. 기준 문장이 없으면 모델은 "긴급"을 자기 상식으로 판단하고, 안전에 관한 글을 긴급으로 보는 쪽으로 기웁니다. 스키마는 `urgent`가 불리언이라는 것만 정합니다. 어떤 경우에 `true`인지는 프롬프트의 기준 문장이 정합니다. 담당팀 정확도가 77건에서 76건으로 바뀐 것은 실행할 때마다 한두 건 바뀌는 범위 안이라 이 실험의 효과로 보기 어렵습니다.
- **본문에 없는 위치.** 세 실행 모두 "본문에 없는 위치"에 걸린 건은 없었습니다. `null`을 허용했고 규칙에 "본문의 글자 그대로"라고 적었기 때문입니다. 걸린 건이 없더라도 이 검사는 남겨 둡니다. 요금 없이 모든 건에 적용할 수 있고, 모델이나 규칙이 바뀌었을 때 지어낸 위치가 생기면 바로 드러납니다.
- **넘길 때 함께 알려 줄 숫자.** 사람이 확인할 건수(`review`와 `manual`), 그리고 `auto` 가운데 틀린 비율입니다. 뒤의 숫자는 정답 라벨이 있어야 계산할 수 있습니다. 실제 업무에서는 80건 모두에 정답을 붙일 수 없으므로 `auto`에서 표본을 뽑아 확인합니다. 이 실습에서는 재실행 비교를 하지 않았습니다. 교재의 예제 2-8처럼 한 번 더 실행해 담당팀이 달라진 건을 `review`에 더하면 `auto`에 남는 오류를 더 줄일 수 있습니다.